In [ ]:
import os
import sys
# block warnings from printing
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import collections
import pandas as pd
import xarray as xr
import cf_xarray as cf
import cftime
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
cartopy.config['data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
cartopy.config['pre_existing_data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns

# settings
%config InlineBackend.figure_format = 'retina'

# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
from map_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from stats_funcs import *

In [ ]:
### +++ DATA PATHS +++ ###

# product keys
keys=['obs','flor']
obs_prods=['merra2']
flor_runs=['ctrl','hicam','hirocky','hitopo']
flor_simNames=['ctrl','cam','hirocky','hitopo']

# store file paths in dictionary
dpath0='/discover/nobackup/projects/giss/baldwin_nip/dmkumar' # top level data directory
opath='/home/dmkumar/JupyterLinks/notebooks/topo_heat/figs'
files={ 'obs'  : { 'merra2' : {} },
        'flor' : { 'ctrl'   : {},
                   'hicam'  : {}, 
                   'hirocky': {},
                   'hitopo' : {} }
      }

for key in ['obs']:
    for i,run in enumerate(obs_prods):
        files[key][run]['slp'] = f'{dpath0}/obs_data/merra2/merra2.SLP.1980-2022.monthly.nc' 

for key in ['flor']:
    for i,run in enumerate(flor_runs):
        simName=flor_simNames[i]
        for fvarn in ['slp']: 
            files[key][run][fvarn] = f'{dpath0}/FLOR/{run}/pi/flor.{simName}.{fvarn}.monthly.nc' 

for key in ['topo']:
    files[key] = {}
    files[key]['etopo'] = f'{dpath0}/topo_files/obs.etopo5.zsurf.nc'
    files[key]['ctrl'] = f'{dpath0}/topo_files/flor.ctrl.zsurf.nc'
    files[key]['hicam'] = f'{dpath0}/topo_files/flor.cam.zsurf.nc'
    files[key]['hitopo'] = f'{dpath0}/topo_files/flor.hitopo.zsurf.nc'

In [ ]:
### +++ ORGANIZE DATA +++ ###

# time bounds
n_years=50
n_keep_days=-1 * (n_years * 365) # number of time steps to keep in days
n_keep_mons=-1 * (n_years * 12)  # number of time steps to keep in months

# initialize dictionaries
dat = { 'obs'  : { 'merra2' : {} },
        'flor' : { 'ctrl'   : {},
                   'hicam'  : {}, 
                   'hirocky': {},
                   'hitopo' : {} }
      }

print('Working on...')
for key in ['obs']:
    varns=['u925','v925']
    print(f'{key}')
    for run in obs_prods:
        for ovarn in ['SLP']:
            varn='slp'
            ds = xr.open_dataset(files[key][run][varn])[ovarn] / 100 # convert from Pa to hPa
            ds_flip = lonFlip(ds) #longitude_flip(ds) # switch lons from -180:180 to 0:360
            dat[key][run][varn] = ds_flip
            del ds
            del ds_flip
        # calculate slp zonal anomalies
        for varn in ['slpa']:
            slp_gm = latitude_weighted_mean(dat[key][run]['slp'])
            dat[key][run][varn] = dat[key][run]['slp']-slp_gm # anomaly relative to lat-weighted global mean
            del slp_gm
            #dat[key][run][varn] = dat[key][run]['slp']-dat[key][run]['slp'].mean(dim='lon') # anomaly relative to zonal mean

for key in ['flor']:
    print(f'{key}')
    for run in flor_runs:
        print(f'...{run}')
        for i,varn in enumerate(['slp']):
            ds = xr.open_dataset(files[key][run][varn], chunks={'time':12})[varn][n_keep_mons:,:,:] # keep only last 50 years  #decode_times=True, use_cftime=True, 
            # update coordinate names to match merra2
            if run in ['hirocky']:
                ds = ds.rename({'grid_xt':'lon','grid_yt':'lat'}) 
            # fix units for metpy
            ds['lat'].attrs['units'] = 'degrees_north'
            ds['lon'].attrs['units'] = 'degrees_east'
            dat[key][run][varn] = ds
            del ds
        # calculate slp anomalies
        for varn in ['slpa']:
            slp_gm = lat_weighted_mean(dat[key][run]['slp'], lat_name='lat', lon_name='lon')
            dat[key][run][varn] = dat[key][run]['slp']-slp_gm # anomaly relative to lat-weighted global mean
            del slp_gm
print('Done.')

topo = {}
print('Loading surface height data.')
for key in ['topo']:
    for case in ['etopo']:
        ds = xr.open_dataset(files[key][case]).ROSE.rename({'ETOPO05_X':'lon', 'ETOPO05_Y':'lat'})
        topo[case] = ds.where(ds>0, np.nan)
        del ds
    for case in ['ctrl', 'hicam', 'hitopo']:
        ds = xr.open_dataset(files[key][case]).ZSURF.rename({'GRID_XT':'lon', 'GRID_YT':'lat'})
        topo[case] = ds.where(ds>0, np.nan)
        del ds

# should be true for North America
topo['hirocky']=topo['hitopo'].where(topo['hitopo'].lat>32, topo['ctrl'])

print('Done.')

In [ ]:
### +++ CALCULATE TIME-MEANS +++ ###
season='JAS'
mons=[7,8,9]
varns=['slp','slpa']

jas_mean = { 'obs' : { 'merra2':{} },
             'flor': { 'ctrl':{}, 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }
ann_jas_mean = { 'obs' : { 'merra2':{} },
                 'flor': { 'ctrl':{}, 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }

print('Calculating seasonal means for...')
for key in ['obs']:
    print(f'{key}')
    for run in obs_prods:
        for varn in varns:
            # seasonal mean for whole timeseries
            custom_seasons = xr.where(dat[key][run][varn]['time'].dt.month.isin(mons), season, 'Other')
            jas_mean[key][run][varn] = dat[key][run][varn].groupby(custom_seasons).mean('time').rename({'month':'season'}).sel(season=season)
            # seasonal mean by year
            ann_jas_mean[key][run][varn] = dat[key][run][varn].sel(time=dat[key][run][varn]['time'].dt.month.isin(mons)).groupby('time.year').mean(dim='time')
      
for key in ['flor']:
    print(f'{key}')
    for run in flor_runs:
        print(f'{run}')
        for varn in varns:
            # seasonal mean for whole timeseries
            custom_seasons = xr.where(dat[key][run][varn]['time'].dt.month.isin(mons), season, 'Other')
            jas_mean[key][run][varn] = dat[key][run][varn].groupby(custom_seasons).mean('time').rename({'month':'season'}).sel(season=season)
            # seasonal mean by year
            ann_jas_mean[key][run][varn] = dat[key][run][varn].sel(time=dat[key][run][varn]['time'].dt.month.isin(mons)).groupby('time.year').mean(dim='time')

print('Done.')

In [ ]:
### +++ COMPARING ALL MODEL RUNS TO OBS. +++ ###

## initialize dictionaries
# for re-gridded obs data
jas_mean_regrid = { 'obs_flor' : {} }
ann_jas_mean_regrid = { 'obs_flor' : {} }
# for significance testing results
obs_diff      = { 'flor' : { 'ctrl':{}, 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }
obs_diff_mask = { 'flor' : { 'ctrl':{}, 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }
obs_ptvals    = { 'flor' : { 'ctrl':{}, 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }

# First, need to put obs data on same grid as model output
regrid_keys = ['obs_flor']

print('Re-gridding obs.')
for i,key in enumerate(['flor']):
    lats=dat[key]['ctrl']['slp'].lat
    lons=dat[key]['ctrl']['slp'].lon
    regrid_key=regrid_keys[i]
    for varn in varns:
        # interpolate obs to CM2.5-FLOR grid
        jas_mean_regrid[regrid_key][varn] = jas_mean['obs']['merra2'][varn].interp(lat=lats, lon=lons, method='linear')
        ann_jas_mean_regrid[regrid_key][varn] = ann_jas_mean['obs']['merra2'][varn].interp(lat=lats, lon=lons, method='linear')
print('Done.\n')

# Determine statistical significance of obs-model differences based on students t-test
print('Significance testing for:')
for key in obs_diff.keys():
    print(f'{key}')
    for run in flor_runs:
        print(f'...{run}')
        for varn in varns:
            diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean_regrid['obs_flor'][varn], ann_jas_mean[key][run][varn], 
                                                   jas_mean_regrid['obs_flor'][varn], jas_mean[key][run][varn])
            obs_diff[key][run][varn] = diff_
            obs_diff_mask[key][run][varn] = diff_mask_
            obs_ptvals[key][run][varn] = ptvals_
print('Done.')

In [ ]:
### +++ COMPARING ALL MODEL RUNS TO MODEL CTRL. +++ ###

# initialize dictionaries
model_diff      = { 'flor' : { 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }
model_diff_mask = { 'flor' : { 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }
model_ptvals    = { 'flor' : { 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }

# names of modified topography runs
flor_mod_runs=['hicam', 'hirocky', 'hitopo']

print('Significance testing for:')
for key in model_diff.keys():
    print(f'{key}')
    for run in flor_mod_runs:
        print(f'...{run}')
        for varn in varns:
            # calculate significance of model - ctrl difference
            diff_, diff_mask_, ptvals_ = sigtest2n(ann_jas_mean[key][run][varn], ann_jas_mean[key]['ctrl'][varn],
                                                   jas_mean[key][run][varn], jas_mean[key]['ctrl'][varn])
            model_diff[key][run][varn] = diff_
            model_diff_mask[key][run][varn] = diff_mask_
            model_ptvals[key][run][varn] = ptvals_

print('Done.')

In [ ]:
### +++ BIAS IMPROVEMENT +++ ###

bias_change        = { 'flor' : { 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }
bias_change_masked = { 'flor' : { 'hicam':{}, 'hirocky':{}, 'hitopo':{} } }

# calculate difference in the absolute value of the model-obs precip difference
# to determine whether or not the change in precipitation is a reduction in the ctrl model bias
for key in bias_change.keys():
    for run in flor_mod_runs:
        for varn in varns:
            bias_change[key][run][varn] = np.abs(obs_diff[key][run][varn])-np.abs(obs_diff[key]['ctrl'][varn])
            bias_change_masked[key][run][varn] = bias_change[key][run][varn].where(model_diff_mask[key][run][varn].mask==False,np.nan)

### SLPA gradient distribution plot

In [ ]:
### +++ DISTRIBUTION OF SLPA GRADIENT +++ ###

slp_dsrt_bnds = [243.5, 247, 32, 35] 
slp_sgoc_bnds = [249.5, 253, 22, 25]
mons=[7,8,9]

# !!!! will need to update to do lat-weighted mean?
# slpa
slp_dsrt = { 'merra2':{}, 'ctrl':{}, 'hicam':{}, 'hirocky':{}, 'hitopo':{} } 
slp_sgoc = { 'merra2':{}, 'ctrl':{}, 'hicam':{}, 'hirocky':{}, 'hitopo':{} }
slp_diff = { 'merra2':{}, 'ctrl':{}, 'hicam':{}, 'hirocky':{}, 'hitopo':{} }

for key in ['obs','flor']:
    if key == 'obs':
        for run in ['merra2']:
            # extract just JAS months
            slpa_jas = dat[key][run]['slpa'].sel(time=dat[key][run]['slpa']['time'].dt.month.isin(mons))
            # average over bounding boxes
            lat_min = 32 ; lat_max = 35 ; lon_min = 243.5 ; lon_max = 247
            slp_dsrt[run] = slpa_jas.sel(lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max)).mean(dim=['lat','lon'])
            lat_min = 22 ; lat_max = 25; lon_min = 249.5 ; lon_max = 253
            slp_sgoc[run] = slpa_jas.sel(lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max)).mean(dim=['lat','lon'])
            # calculate slp difference
            slp_diff[run] = slp_sgoc[run] - slp_dsrt[run]
    elif key == 'flor':
        for run in ['ctrl','hicam','hirocky','hitopo']:
            # extract just JAS months
            slpa_jas = dat[key][run]['slpa'].sel(time=dat[key][run]['slpa']['time'].dt.month.isin(mons))
            # average over bounding boxes
            lat_min = 32 ; lat_max = 35 ; lon_min = 243.5 ; lon_max = 247
            slp_dsrt[run] = slpa_jas.sel(lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max)).mean(dim=['lat','lon'])
            lat_min = 22 ; lat_max = 25; lon_min = 249.5 ; lon_max = 253
            slp_sgoc[run] = slpa_jas.sel(lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max)).mean(dim=['lat','lon'])
            # calculate slp difference
            slp_diff[run] = slp_sgoc[run] - slp_dsrt[run]
    else:
        pass
    
# create dataframes
slp_df = pd.concat(
    {model: da.to_series() for model, da in slp_diff.items()},
    axis=1
)


In [ ]:
colors = ['black','lightsteelblue','peru','coral','darkred']
f, ax = plt.subplots(ncols=1, figsize=(9,5))

for col, c in zip(slp_df.columns, colors):
    sns.kdeplot(
        data=slp_df[col],
        ax=ax,
        fill=True,
        color=c,
        label=col
    )
ax.set_xlabel('SLP anomaly difference magnitude')
ax.set(xlim=[-4,8])
ax.legend()

### Maps of SLPA change and bias

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array([r'MERRA2$-$CTRL', r'HI$_{\mathbf{MEX}}$$-$CTRL', r'HI$_{\mathbf{ROCKY}}$$-$CTRL', r'HI$_{\mathbf{GBL}}$$-$CTRL'])
letters=['A','B','C','D']
tx=-110.5
ty=40.25
# var specs
lat=jas_mean['flor']['ctrl']['slp'].lat
lon=jas_mean['flor']['ctrl']['slp'].lon
# vector specs
skip_nh=3
skip_nl=1
w=0.006
scalef=2
key_length=1
# bias colormap
dcmap=cm.RdBu_r
dvmin=-3
dvmax=3
dlevels=np.linspace(dvmin, dvmax, 25)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# topography contours
zlevels=np.linspace(1000,3000,7)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[239,260,15,40]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(20,7), layout='constrained', subplot_kw={'projection':proj})

#fig.text(.5,1.0,f'Modified Topo $-$ CTRL', **text_kw)
fig.text(0.025,0,
         'Masked for statistical significance. Hatching indicates where higher topography increased model biases (calculated as MERRA2$-$Model).\n'
         'Thick black lines are model surface height boundary conditions, intervals of 500 m starting at 1 km.',
         **text_kw3)

#=== CTRL ===
cf=ax[0].pcolormesh(lon, lat, obs_diff_mask['flor']['ctrl']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[0].contour(lon, lat, topo['ctrl'], levels=zlevels, linewidths=2, colors='black', transform=trans)

#=== HI-CAM ===
ax[1].pcolormesh(lon, lat, model_diff_mask['flor']['hicam']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[1].contour(lon, lat, topo['hicam'], levels=zlevels, linewidths=2, colors='black', transform=trans)

#=== HI-ROCKY ===
ax[2].pcolormesh(lon, lat, model_diff_mask['flor']['hirocky']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[2].contour(lon, lat, topo['hirocky'], levels=zlevels, linewidths=2, colors='black', transform=trans)

#=== HI-GBL ===
ax[3].pcolormesh(lon, lat, model_diff_mask['flor']['hitopo']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[3].contour(lon, lat, topo['hitopo'], levels=zlevels, linewidths=2, colors='black', transform=trans)

for i, ax in enumerate(ax.flat): 
    # add box around core NAM domain
    lon_bnds = np.array([-113, -104, -104, -113])
    lat_bnds = np.array([20,  20,  34,  34])
    ring=LinearRing(list(zip(lon_bnds, lat_bnds)))
    #ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=3, linestyle='--', zorder=11) 
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-121, ty+0.5, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=.75)
    ax.add_feature(cfeature.STATES, edgecolor='k', linewidth=.75)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=.75)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# colorbar
cax=fig.add_axes([1.01, 0.15, 0.025, 0.7])
cbar=fig.colorbar(cf, ticks=np.linspace(-10,10,21), orientation='vertical', extend='both', cax=cax)
cbar.set_label('$\Delta$SLP (rel. to global mean) [hPa]', labelpad=25, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

#plt.savefig(f'{opath}/slp.925wind.change.flor.{season}.pdf', transparent=False, bbox_inches='tight')

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array([r'MERRA2$-$CTRL', r'MERRA2$-$HI$_{\mathbf{MEX}}$', r'MERRA2$-$HI$_{\mathbf{ROCKY}}$', r'MERRA2$-$HI$_{\mathbf{GBL}}$'])
letters=['A','B','C','D']
tx=-110.5
ty=40.25
# var specs
lat=jas_mean['flor']['ctrl']['slp'].lat
lon=jas_mean['flor']['ctrl']['slp'].lon
# vector specs
skip_nh=3
skip_nl=1
w=0.006
scalef=2
key_length=1
# bias colormap
dcmap=cm.RdBu_r
dvmin=-3
dvmax=3
dlevels=np.linspace(dvmin, dvmax, 25)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# topography contours
zlevels=np.linspace(1000,3000,7)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[239,260,15,40]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(20,7), layout='constrained', subplot_kw={'projection':proj})

#fig.text(.5,1.0,f'Modified Topo $-$ CTRL', **text_kw)
fig.text(0.025,0,
         'Masked for statistical significance. Hatching indicates where higher topography increased model biases (calculated as MERRA2$-$Model).\n'
         'Thick black lines are model surface height boundary conditions, intervals of 500 m starting at 1 km.',
         **text_kw3)

#=== CTRL ===
cf=ax[0].pcolormesh(lon, lat, obs_diff_mask['flor']['ctrl']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[0].contour(lon, lat, topo['ctrl'], levels=zlevels, linewidths=2, colors='black', transform=trans)

#=== HI-CAM ===
ax[1].pcolormesh(lon, lat, obs_diff_mask['flor']['hicam']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[1].contour(lon, lat, topo['hicam'], levels=zlevels, linewidths=2, colors='black', transform=trans)

#=== HI-ROCKY ===
ax[2].pcolormesh(lon, lat, obs_diff_mask['flor']['hirocky']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[2].contour(lon, lat, topo['hirocky'], levels=zlevels, linewidths=2, colors='black', transform=trans)

#=== HI-GBL ===
ax[3].pcolormesh(lon, lat, obs_diff_mask['flor']['hitopo']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[3].contour(lon, lat, topo['hitopo'], levels=zlevels, linewidths=2, colors='black', transform=trans)

for i, ax in enumerate(ax.flat): 
    # add box around core NAM domain
    lon_bnds = np.array([-113, -104, -104, -113])
    lat_bnds = np.array([20,  20,  34,  34])
    ring=LinearRing(list(zip(lon_bnds, lat_bnds)))
    #ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=3, linestyle='--', zorder=11) 
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-121, ty+0.5, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=.75)
    ax.add_feature(cfeature.STATES, edgecolor='k', linewidth=.75)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=.75)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# colorbar
cax=fig.add_axes([1.01, 0.15, 0.025, 0.7])
cbar=fig.colorbar(cf, ticks=np.linspace(-10,10,21), orientation='vertical', extend='both', cax=cax)
cbar.set_label('$\Delta$SLP (rel. to global mean) [hPa]', labelpad=25, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

#plt.savefig(f'{opath}/slp.925wind.change.flor.{season}.pdf', transparent=False, bbox_inches='tight')

### map of impact of Baja

In [ ]:
baja_impact_mask = model_diff_mask['flor']['hitopo']['slpa'] - (model_diff_mask['flor']['hicam']['slpa']+model_diff_mask['flor']['hirocky']['slpa'])
baja_impact = model_diff['flor']['hitopo']['slpa'] - (model_diff['flor']['hicam']['slpa']+model_diff['flor']['hirocky']['slpa'])

In [ ]:
na_mask = topo['hirocky'].where(topo['hirocky'].lat>32, np.nan)
mex_mask = topo['hicam'].where( (topo['hicam'].lat < 32) & (topo['hicam'].lon > 247), np.nan)
baja_topo = topo['hitopo'].where(na_mask.isnull() & mex_mask.isnull()) # rough way to isolate Baja

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
titles=np.array([r'HI$_{\mathbf{MEX}}$$-$CTRL', r'HI$_{\mathbf{ROCKY}}$$-$CTRL', r'HI$_{\mathbf{GBL}}$$-$CTRL', r'HI$_{\mathbf{GBL}}$$-$(HI$_{\mathbf{MEX}}$ + HI$_{\mathbf{ROCKY}}$)'])
letters=['A','B','C','D']
tx=-110.5
ty=40.25
# var specs
lat=jas_mean['flor']['ctrl']['slp'].lat
lon=jas_mean['flor']['ctrl']['slp'].lon
# vector specs
skip_nh=3
skip_nl=1
w=0.006
scalef=2
key_length=1
# bias colormap
dcmap=cm.RdBu_r
dvmin=-3
dvmax=3
dlevels=np.linspace(dvmin, dvmax, 25)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# topography contours
zlevels=np.linspace(500,3500,7)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[239,260,15,40]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(20,6), layout='constrained', subplot_kw={'projection':proj})

fig.text(0,0,
         'Not masked for statistical significance. Thick black lines are model surface height boundary conditions, intervals of 500 m starting at 500 m.',
         **text_kw3)

#=== HI-CAM ===
ax[0].pcolormesh(lon, lat, model_diff['flor']['hicam']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[0].contour(lon, lat, topo['hicam'], levels=zlevels, linewidths=2, colors='black', transform=trans)

#=== HI-ROCKY ===
ax[1].pcolormesh(lon, lat, model_diff['flor']['hirocky']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[1].contour(lon, lat, topo['hirocky'], levels=zlevels, linewidths=2, colors='black', transform=trans)

#=== HI-GBL ===
ax[2].pcolormesh(lon, lat, model_diff['flor']['hitopo']['slpa'], cmap=dcmap, norm=dnorm, transform=trans)
ax[2].contour(lon, lat, topo['hitopo'], levels=zlevels, linewidths=2, colors='black', transform=trans)

#=== IMPACT OF BAJA ===
ax[3].pcolormesh(lon, lat, baja_impact, cmap=dcmap, norm=dnorm, transform=trans)
ax[3].contour(lon, lat, baja_topo, levels=zlevels, linewidths=2, colors='black', transform=trans)


for i, ax in enumerate(ax.flat): 
    # add box around core NAM domain
    lon_bnds = np.array([-113, -104, -104, -113])
    lat_bnds = np.array([20,  20,  34,  34])
    ring=LinearRing(list(zip(lon_bnds, lat_bnds)))
    #ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=3, linestyle='--', zorder=11) 
    # subplot labels
    ax.text(tx, ty, titles[i], **text_kw)
    ax.text(-121, ty+0.5, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=.75)
    ax.add_feature(cfeature.STATES, edgecolor='k', linewidth=.75)
    ax.add_feature(cfeature.BORDERS, edgecolor='k', linewidth=.75)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# colorbar
cax=fig.add_axes([1.01, 0.15, 0.025, 0.7])
cbar=fig.colorbar(cf, ticks=np.linspace(-10,10,21), orientation='vertical', extend='both', cax=cax)
cbar.set_label('$\Delta$SLP (rel. to global mean) [hPa]', labelpad=25, rotation=270, size=18, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=18)
for tick in cbar.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('normal')

#plt.savefig(f'{opath}/slp.925wind.change.flor.{season}.pdf', transparent=False, bbox_inches='tight')